# Lab 22: Audio Language Models: Speech In, Speech Out

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
In Labs 17 and 18 the Jetson heard with **Whisper** and spoke with **Piper**. Each of them does one job: Whisper only turns sound into text, Piper only turns text into sound. Neither of them can *think*. In this lab you use an **audio language model**: one language model that has **ears** and a **voice**. It takes sound as input (a spoken question), thinks like the language model of Lab 20, and can give its answer as text **or** as speech. The model is **LFM3-Audio-2B** from Liquid AI, a 2-billion-parameter model that runs on your Jetson with llama.cpp, with no internet and no account. You test how well it listens, how well it speaks, and how fast it is, you compare it with Whisper and Piper, and you build a small voice responder.

**By the end of this lab you can:**
1. Explain what an audio language model is and how it differs from a speech-to-text model plus a text-to-speech model.
2. Start the audio model on the Jetson and turn speech into text with a system prompt.
3. Measure the speed (real-time factor) and the accuracy (word error rate) of the model.
4. Give the model a spoken question and read its answer, and make it speak with a female or a male voice in several languages.
5. Build a voice responder that listens, thinks and speaks, and say where this model is weaker than Whisper and Piper.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. A sound player appears under the cell that makes sound: press its play button. Press **Ctrl + S** often.

> **Memory.** The audio model (the model file and its ears and voice file) uses about 1.5 GB. It does **not** fit together with the text model of Lab 20 (about 5 GB) and the other students' work, so in Part A **one** student per Jetson stops the text model and starts the audio server; everybody else uses it. In the last step the text model is brought back.
> **Privacy.** The microphone hears people. Record only yourself and classmates who agree. The sound is processed on the Jetson and is not sent to the internet.
> **Be critical.** The model can write words that nobody said, and it speaks well only in the languages it was trained on.

## 2. Background

**One model for all.** In the multimodal pipeline of Lab 23 there is a different small model for each kind of data. An **audio language model** puts the ears and the voice *into the language model*:

```
speech --> audio encoder (the ears) --> tokens --+
                                                 +--> language model --> text tokens --> text
your text ------> text tokens -------------------+                  \--> speech tokens --> speech decoder (the voice) --> speech
```

1. The **audio encoder** cuts the sound into short frames (80 ms) and turns each frame into a list of numbers. A projector turns them into the kind of numbers that the language model uses for words (the same idea as the eyes of Lab 21).
2. The **language model** reads the audio tokens as if they were words, and writes either **text tokens** (an answer or a transcript) or **speech tokens**.
3. The **speech decoder** turns the speech tokens into a waveform again, in a male or a female voice.

The ears and the voice are stored in a second file, the **mmproj** file, next to the model file (as the eyes in Lab 21). The model file is 1.1 GB, the mmproj file 0.3 GB.

**What the system prompt does.** The same model does different jobs, chosen by the first message:

| System prompt | Job |
|---|---|
| `Perform ASR.` | Speech recognition: write what was said. The language is found by the model. |
| `Perform ASR in French.` | Speech recognition in a named language. |
| `Answer in one short sentence.` | The spoken sound is a **question**; the answer is text. |

**Whisper and Piper, or one audio model?**

| | Whisper (Lab 17) + Piper (Lab 18) | LFM3-Audio-2B |
|---|---|---|
| Parts | Two models, one for each job | One model for listening, thinking and speaking |
| Thinking | None; a text model is needed (Lab 20) | Built in: it can answer a spoken question |
| Memory | Small (Whisper `base.en` about 0.3 GB, a Piper voice 60 MB) | About 1.5 GB |
| Speed | Whisper `base.en` about 0.6 of real time on the CPU; Piper fast | Listening about 0.3 to 0.5 of real time on the GPU; speaking is about as fast as real time (a short sentence takes 2 to 4 s) |
| Languages | Whisper: about 100 (with Malay); Piper: one file for each voice | 18 languages; **no Malay** |
| Control | Many settings (model size, speed, pitch) | A system prompt and a voice (female or male) |

**Real-time factor and word error rate.** As in Labs 17 and 18: `RTF = time it took / length of the sound` (below 1 is faster than real time), and the **word error rate** (WER) is the share of words that are wrong: `(substitutions + deletions + insertions) / words said`.

**Why speech output is slower than Piper.** Text is written one token at a time. Speech is written one **frame** (80 ms of sound) at a time, and one frame needs several steps of the model. So the voice of a language model needs about as long as the sound lasts, while a small dedicated voice such as Piper is made much faster than it is spoken.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(22)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (25 min)

### Step 1: Check the Parts
The cell checks the programs, the sound helpers and the memory. It also shows which model server runs now.

In [ ]:
import csv
import glob
import os
import re
import shutil
import subprocess
import sys
import time
from datetime import datetime

import numpy as np

import labview

print("This notebook is saved in:", os.getcwd())
print("llama-server:", "OK" if os.path.exists(labview.LLM_SERVER) else "MISSING (ask the teacher to run labs-setup.sh)")
print("Text model server running:", labview.llm_ready(), "-", labview.llm_current())
print("Audio server running:     ", labview.alm_ready())
free = int(subprocess.run("free -m | awk 'NR==2{print $7}'", shell=True, capture_output=True, text=True).stdout.strip() or 0)
print(f"Memory available: {free} MB (the audio model needs about 1500 MB)")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which model server runs now:
- How much memory is available:

### Step 2: Start the Audio Server (only the ONE student that the teacher chose)
Ask your team first: the **text model of Lab 20 stops** for everybody on this Jetson, because the two models do not fit together. The chosen student runs the cell. It stops the text model, downloads the two files the first time (about 1.4 GB, a few minutes) and starts `llama-server` with the ears and the voice (`--mmproj`). The other students do **not** run it: they wait for the message *Audio server ready*, then run the check cell.

In [ ]:
if labview.llm_ready():
    labview.llm_stop()                 # frees about 5 GB (tell your team first)
labview.alm_start()

In [ ]:
print("Audio server ready:", labview.alm_ready())
print("Text model server running:", labview.llm_ready())

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How long did the server take to start:
- Why can the text model of Lab 20 and the audio model not run together:

### Step 3: Choose the Microphone and the Speaker
Run `labview.audio_devices()`, as in Labs 17 and 18, and set your microphone. The sound player under a cell always plays on your **laptop**.

In [ ]:
labview.audio_devices()

In [ ]:
MIC = "plughw:2,0"          # change this to your microphone
labview.mic_set(MIC)

print("Say 'one two three' after the message ...")
time.sleep(1)
print(">>> SPEAK NOW <<<")
sound = labview.record(3)
print("Done. Samples:", len(sound), "seconds:", round(len(sound) / 16000, 1))
labview.play(sound)

---
# Part B: Listen: Speech to Text (45 min)

### Step 1: Your First Transcript
The helper `labview.alm_listen(sound)` sends the sound to the audio server (as a WAV file inside an HTTP request, like the picture of Lab 21) and returns the text, the time it took and the real-time factor. The system prompt `Perform ASR.` tells the model to write what was said.

In [ ]:
print("Get ready ...")
time.sleep(1)
print(">>> SPEAK NOW: The Jetson can hear and understand me. <<<")
sound = labview.record(5)
result = labview.alm_listen(sound)
print()
print("Heard:", repr(result["text"]))
print(f"Took {result['seconds']:.1f} s for {result['audio_seconds']:.1f} s of sound, real-time factor {result['rtf']:.2f}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- What the model heard, and what you said:
- The real-time factor, and what it means:
- Run the cell again: is the first run slower than the second? Why could that be:

### Step 2: Accuracy: the Word Error Rate
As in Lab 17, the function `wer` counts how many words are wrong (the edit distance between the words said and the words heard). Say each sentence and read the result.

In [ ]:
def words_of(text):
    return re.findall(r"[a-z0-9']+", text.lower())


def wer(said, heard):
    """Word error rate: the edit distance between the words said and the words heard, divided by the number of words said."""
    a, b = words_of(said), words_of(heard)
    d = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        previous, d[0] = d[0], i
        for j in range(1, len(b) + 1):
            previous, d[j] = d[j], min(d[j] + 1, d[j - 1] + 1, previous + (a[i - 1] != b[j - 1]))
    return d[len(b)] / max(len(a), 1)


SENTENCES = [
    "turn on the green light",
    "what is the temperature in the room",
    "the quick brown fox jumps over the lazy dog",
    "please count the people at the door",
    "start the camera and stop the music",
]

scores, seconds = [], []
for sentence in SENTENCES:
    print()
    print(f">>> SPEAK NOW: {sentence} <<<")
    result = labview.alm_listen(labview.record(5))
    scores.append(wer(sentence, result["text"]))
    seconds.append(result["seconds"])
    print("Heard:", repr(result["text"]), f"-> WER {100 * scores[-1]:.0f} %, {result['seconds']:.1f} s")
print()
print(f"Average WER: {100 * sum(scores) / len(scores):.0f} %   average time: {sum(seconds) / len(seconds):.1f} s")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Average WER of the audio model:
- Which sentence was the hardest, and why:
- Your WER with Whisper in Lab 17 (look in your Lab 17 notebook), and which model was better:

### Step 3: Compare with Whisper on the Same Sound
Both models must hear **exactly the same** recording to be compared fairly. The cell records one sentence, gives it to the audio model, and, if Whisper is installed (Lab 17), also to Whisper `base.en`. It prints both transcripts and both real-time factors.

In [ ]:
try:
    from faster_whisper import WhisperModel
    whisper = WhisperModel("base.en", device="cpu", compute_type="int8", cpu_threads=4)
except Exception as problem:
    whisper = None
    print("Whisper is not available here (", type(problem).__name__, "). The cell shows only the audio model.")

TEST = "please count the people at the door"
print(f">>> SPEAK NOW: {TEST} <<<")
sound = labview.record(5)

alm = labview.alm_listen(sound)
print(f"Audio model: {alm['text']!r}  WER {100 * wer(TEST, alm['text']):.0f} %  {alm['seconds']:.1f} s  RTF {alm['rtf']:.2f}")
if whisper:
    started = time.time()
    segments, _ = whisper.transcribe(sound.astype(np.float32), language="en", beam_size=1)
    text = " ".join(s.text.strip() for s in segments).strip()
    took = time.time() - started
    print(f"Whisper:     {text!r}  WER {100 * wer(TEST, text):.0f} %  {took:.1f} s  RTF {took / (len(sound) / 16000):.2f}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which model was more accurate on this sound:
- Which model was faster, and what is the difference in real-time factor:
- Name one reason to choose Whisper and one reason to choose the audio model:

### Step 4: Other Languages
LFM3-Audio was trained on 18 languages: English, Arabic, Danish, German, Greek, Spanish, Finnish, French, Hebrew, Italian, Japanese, Korean, Dutch, Polish, Portuguese, Swedish, Turkish and Chinese. **Malay is not on the list.** Try the languages you know. With `Perform ASR.` the model finds the language itself; with `Perform ASR in French.` you name it.

In [ ]:
LANGUAGE = "Malay"          # try: English, French, Spanish, German, Japanese, Chinese ... and Malay (not trained)
print(f">>> SPEAK NOW in {LANGUAGE}: any short sentence <<<")
sound = labview.record(5)
for system in ("Perform ASR.", f"Perform ASR in {LANGUAGE}."):
    result = labview.alm_listen(sound, system=system)
    print(f"{system:<28} -> {result['text']!r}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The language you tried, what you said and what the model wrote:
- What happened with Malay, and why:
- Whisper in Lab 17 can also do Malay. When would you choose Whisper:

---
# Part C: Understand: A Spoken Question (30 min)

### Step 1: Ask by Voice
So far the model only wrote down the words. Now change the system prompt: the sound is a **question**, and the model **answers** it. There is no text between your voice and the answer: the audio tokens go straight into the language model. Ask a short question that does not need facts that the model could not know, for example *How many days are in a week?* or *Name three colours.*

In [ ]:
SYSTEM = "Answer in one short sentence."
print(">>> ASK A QUESTION NOW <<<")
sound = labview.record(5)
heard = labview.alm_listen(sound)["text"]
answer = labview.alm_listen(sound, system=SYSTEM)
print("You said (Perform ASR):", repr(heard))
print("The model answers:     ", repr(answer["text"]))
print(f"Took {answer['seconds']:.1f} s")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Your question and the answer:
- Is the answer right? Did the model understand the question:

### Step 2: One Step or Two?
There are two ways to answer a spoken question:

- **Two steps:** the model first writes down the words (`Perform ASR.`), then answers the text.
- **One step:** the model answers the sound directly (as in Step 1).

The cell measures both ways for the same recording. A spoken answer needs a good guess of what was said, so a mistake in step one gives a wrong answer in step two.

In [ ]:
print(">>> ASK A QUESTION NOW <<<")
sound = labview.record(5)

started = time.time()
heard = labview.alm_listen(sound)
text_answer = labview.alm_ask(heard["text"], system=SYSTEM)
two = time.time() - started

started = time.time()
one_answer = labview.alm_listen(sound, system=SYSTEM)
one = time.time() - started

print("Two steps:", repr(text_answer["text"]), f"({two:.1f} s)")
print("One step: ", repr(one_answer["text"]), f"({one:.1f} s)")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The two answers and the two times:
- Which way is faster, and why:
- Which way would you choose for a robot that must answer at once:

### Step 3: The Limits
A small model can be wrong in new ways when it hears. Try three things and write what happens: a **quiet** whisper, a question in **noise** (a classmate talks, music plays), and a question about **this class** (*Who is the teacher?*). Remember Lab 23: the model has no notes about your class.

In [ ]:
TRIES = ["a whispered question", "a question with a classmate talking", "Who is the teacher of this class?"]
for situation in TRIES:
    print()
    print(f">>> SPEAK NOW ({situation}) <<<")
    sound = labview.record(5)
    heard = labview.alm_listen(sound)["text"]
    answer = labview.alm_listen(sound, system=SYSTEM)["text"]
    print(f"{situation}\n   heard : {heard!r}\n   answer: {answer!r}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- What went wrong in noise or in a whisper:
- What did the model do with the question about the class:
- What would fix the last problem (think of Lab 24):

---
# Part D: Speak: Text to Speech (40 min)

### Step 1: Your First Voice
`labview.alm_speak(text, path, voice, lang)` sends the text to the audio server (a request to `/tts`). The server makes the sound with the speech decoder that is in the mmproj file, and the helper saves it as a WAV file at 24 000 samples per second. The voice is `female` or `male`. Look at the times.

In [ ]:
TEXT = "Hello. I am the Jetson Orin Nano. I can speak now."
for voice in ("female", "male"):
    took = labview.alm_speak(TEXT, f"alm_{voice}.wav", voice=voice)
    sound, rate = labview.load_wav(f"alm_{voice}.wav")
    seconds = len(sound) / rate
    print(f"{voice}: {seconds:.1f} s of sound at {rate} Hz, made in {took:.1f} s, real-time factor {took / seconds:.2f}")
    labview.play(f"alm_{voice}.wav")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The two voices: how do they sound:
- The real-time factor, and what it means for a voice that must answer at once:
- The real-time factor of this voice compared with the one of Piper in Lab 18:

### Step 2: Languages
`lang` takes a language code (`en`, `fr`, `es`, `de`, `ja`, ...). Make the same sentence in two languages that the model knows, and in Malay, which it does not know. Listen to each.

In [ ]:
SENTENCES = {
    "en": "Good morning. The room is open.",
    "fr": "Bonjour. La salle est ouverte.",
    "es": "Buenos días. La sala está abierta.",
    "ms": "Selamat pagi. Bilik ini sudah dibuka.",
}
for lang, text in SENTENCES.items():
    took = labview.alm_speak(text, f"alm_{lang}.wav", lang=lang)
    sound, rate = labview.load_wav(f"alm_{lang}.wav")
    print(f"{lang}: {text}  ({len(sound) / rate:.1f} s of sound, made in {took:.1f} s)")
    labview.play(f"alm_{lang}.wav")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which languages sounded right:
- How did Malay sound, and why:
- Which voice of Lab 18 would you use for Malay:

### Step 3: Compare with Piper
Listen to the same sentence made by Piper (Lab 18) and by the audio model. The cell makes the Piper voice only if you did Lab 18 on this Jetson (the voice file is in `voices/`). Score each voice from 1 (bad) to 5 (very good) for **clearness**, **naturalness** and **speed**.

In [ ]:
TEST = "Please count the people at the door. Two people are inside."
took = labview.alm_speak(TEST, "alm_test.wav")
sound, rate = labview.load_wav("alm_test.wav")
print(f"Audio model: {len(sound) / rate:.1f} s of sound, made in {took:.1f} s")
labview.play("alm_test.wav")

voice_file = "../lab18/voices/en_US-lessac-medium.onnx"
if os.path.exists(voice_file):
    started = time.time()
    subprocess.run([sys.executable, "-m", "piper", "-m", voice_file, "-f", "piper_test.wav"], input=TEST.encode(), capture_output=True)
    took = time.time() - started
    sound, rate = labview.load_wav("piper_test.wav")
    print(f"Piper:       {len(sound) / rate:.1f} s of sound, made in {took:.1f} s")
    labview.play("piper_test.wav")
else:
    print("The Piper voice of Lab 18 is not here, so only the audio model speaks.")

| Voice | Clearness (1 to 5) | Naturalness (1 to 5) | Speed (seconds to make it) |
|---|---|---|---|
| Piper (Lab 18) | | | |
| LFM3-Audio female | | | |
| LFM3-Audio male | | | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which voice you prefer, and why:
- When would you use Piper, and when the audio model:

---
# Part E: Build a Voice Responder (30 min)

### Step 1: Listen, Think, Speak
Now put the three parts together: a **responder** that listens to your question (the audio model), answers (the same model), and speaks the answer (the `/tts` request). The responder writes every turn in a log file. The cell shows the time of each part: this is the **latency** that you know from Lab 23.

In [ ]:
# ---------------- Settings you can change ----------------
SECONDS = 5                    # the longest question
STOP_WORD = "goodbye"          # say this word to end the conversation
SYSTEM = "You are a short and friendly lab helper. Answer in one or two short sentences."
VOICE = "female"
# ----------------------------------------------------------


def log_turn(heard, answer, t_listen, t_think, t_speak):
    first = not os.path.exists("voice_log.csv")
    with open("voice_log.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "heard", "answer", "listen_s", "think_s", "speak_s"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), heard, answer, round(t_listen, 1), round(t_think, 1), round(t_speak, 1)])


def one_turn(sound):
    """Listen, think and speak for one question. Returns the text that was heard, or None when nothing was said."""
    started = time.time()
    heard = labview.alm_listen(sound)["text"]
    t_listen = time.time() - started
    if not heard:
        print("Nobody spoke.")
        return None
    started = time.time()
    answer = labview.alm_listen(sound, system=SYSTEM)["text"]
    t_think = time.time() - started
    started = time.time()
    labview.alm_speak(answer, "answer.wav", voice=VOICE)
    t_speak = time.time() - started
    print(f"You    : {heard}\nHelper : {answer}\n[listen {t_listen:.1f} s | think {t_think:.1f} s | speak {t_speak:.1f} s]")
    labview.play("answer.wav")
    log_turn(heard, answer, t_listen, t_think, t_speak)
    return heard


print("The responder is ready. Run the next cell to talk to it.")

In [ ]:
print(f"Ask a question. Say '{STOP_WORD}' to stop. Press the Stop button of the toolbar to stop at once.")
while True:
    print(">>> SPEAK NOW <<<")
    heard = one_turn(labview.record(SECONDS))
    if heard and STOP_WORD in heard.lower():
        print("Goodbye.")
        break

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which part of the turn took the longest:
- How long did a whole turn take, from the end of your question to the first sound of the answer:
- Is that fast enough for a conversation? What would you change:

---
# Part F: Make It Your Own, Test and Present (35 min)

### Step 1: Make It Your Own
Do **at least one** of these, or one of your own. Change the code in the cells above, run that cell again, then try again.

- Give the responder a **character** with another system prompt (a pirate, a strict teacher) and listen to the voice.
- Make it answer in **French** or **Spanish**: speak the question in that language and set `lang` in `alm_speak`.
- Add a **command**: if the heard text contains *light on* or *time*, do the action in Python (as in Lab 17) and only ask the model for the other questions.
- Measure the **word error rate** in a **noisy** place and with the microphone far away.
- Compare the audio model with Whisper and Piper in a table of **speed**, **memory** and **accuracy**.
- Make the responder **skip** a turn when the sound is too quiet (use the level function of Lab 17).

> **Tip.** Change one thing at a time. To keep a copy of a cell before a big change, copy the cell (select it, press **C**, then **V**) and change the copy.

### Step 2: Test
Run the responder 5 times with different questions. Fill in the table.

| # | Question you asked | Heard correctly? (yes/no) | Answer right? (yes/no) | Time of the turn (s) |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- What is the most common problem:
- What did you change to fix it, and did it work:

### Step 3: Report and Present
Write a short report (half a page): what an audio language model is, what you built, the measured numbers (real-time factor, word error rate, time of a turn) and what you learned. Present it to your instructor in 3 minutes and show the responder.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Model, voice and system prompt used:
- The most important number you measured:

### Step 4: Bring the Text Model Back
Before the next lab, the chosen student stops the audio server and starts the text model again. Tell your team first.

In [ ]:
labview.alm_stop()
labview.llm_start()

> **Where next.** The Jetson can now hear and speak with **one** model. In Lab 23 you join *all* the senses (wake word, ears, eyes, brain and voice) in a multi-modal assistant, using the multi-modal model **LFM3-Omni**. In Lab 24 you give the language model notes so that it answers from facts (RAG).

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The text model is stopped, the audio server runs and answers `labview.alm_ready()` | |
| 2 | A recorded sentence is turned into text with `Perform ASR.` and the real-time factor is read | |
| 3 | The word error rate of 5 sentences is measured and compared with Whisper | |
| 4 | The model answers a spoken question, and you compare one step with two steps | |
| 5 | Both voices (female and male) speak, and the languages are tried (including Malay) | |
| 6 | The voice score table is complete | |
| 7 | The voice responder listens, answers and speaks, and the log file shows the times | |
| 8 | At least one improvement works and you explain the code you changed | |
| 9 | The test table is complete | |
| 10 | The text model is started again, and the report and the 3-minute presentation are given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- 1. What is an audio language model? How is it different from Whisper plus Piper?:
- 2. What do the ears and the voice of the model do, and in which file are they?:
- 3. What does the system prompt `Perform ASR.` do? What happens if you change it?:
- 4. What are the real-time factor and the word error rate?:
- 5. Why is the voice of a language model slower than Piper?:
- 6. Name one advantage and one disadvantage of using one model for hearing, thinking and speaking.:
- 7. Why can the text model of Lab 20 not run at the same time as the audio model?:
- 8. Why does the model not speak Malay well? What would you use instead?:
- 9. Which is the best use of an audio language model? (a) Answering a spoken question with a spoken answer (b) Making the camera faster (c) Training a wake word (d) Saving memory:

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `The text model is running and the memory is not enough for both` | The text model of Lab 20 still runs | Tell your team, then run `labview.llm_stop()` and start Step 2 again |
| `Cannot reach the audio server` | The audio server is not running | The chosen student runs `labview.alm_start()` |
| The audio server did not start in time | The files are still downloading, or the memory is low | Run the cell again. Look at the log: `!tail -20 ~/audio-server.log` |
| `cudaMalloc failed: out of memory` in the log | Too many programs, or a big file was just copied | Close other notebooks (Kernel, Shut Down All Kernels). Ask the teacher to restart the Jetson. |
| `The audio server could not make the speech` | The audio server is not running or has stopped | The chosen student runs `labview.alm_start()`. Look at the log: `!tail -20 ~/audio-server.log` |
| The transcript is empty | The sound is too quiet, or the microphone is wrong | Run `labview.audio_devices()`, set `MIC`, speak louder and closer |
| The transcript has words nobody said | The model invents words from noise | Record in a quiet place, or skip quiet sounds |
| The Malay speech is wrong | Malay is not a language of the model | Use espeak-ng with `voice="ms"` (Lab 18) |
| The text model does not come back after the lab | Step 4 was not run | Run `labview.alm_stop()` and `labview.llm_start()` |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(22)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- LFM3-Audio-2B (GGUF): https://huggingface.co/LiquidAI/LFM3-Audio-2B-GGUF
- Liquid AI on llama.cpp: https://docs.liquid.ai/deployment/on-device/llama-cpp
- Whisper (Lab 17): https://github.com/openai/whisper
- Word error rate: https://en.wikipedia.org/wiki/Word_error_rate